# Chapter 39: IMU Modeling

<a href="../lite/lab/index.html?path=ch39_imu_modeling.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite — run and edit this notebook</a>

*Runs entirely in your browser; no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

An IMU measures acceleration and rotation rate 200 times per second. Integrate once for velocity, twice for position. Simple, right? Except there is a tiny bias, maybe 0.01 m/s$^2$, hiding in the accelerometer. After 10 seconds, that bias has created a 0.5 m position error. After 60 seconds, 18 meters. After 5 minutes, 450 meters. The IMU is your best friend for short term and your worst enemy for long term.

This chapter builds a complete IMU simulation from first principles, shows exactly why drift happens, and demonstrates what you can (and cannot) do about it.

---
## 39.1 What an IMU Actually Measures

An Inertial Measurement Unit contains two sensors:

| Sensor | Measures | Units |
|--------|----------|-------|
| **Accelerometer** | Specific force (acceleration minus gravity) | m/s$^2$ |
| **Gyroscope** | Angular velocity | rad/s |

The **accelerometer** does *not* measure acceleration directly. It measures **specific force**: the non-gravitational acceleration experienced by the sensor. When the IMU sits still on a table, the accelerometer reads $+g$ upward (the table pushing against gravity), *not* zero.

$$\mathbf{a}_{\text{meas}} = R^\top(\mathbf{a}_{\text{true}} - \mathbf{g}) + \mathbf{b}_a + \mathbf{n}_a$$

The **gyroscope** measures angular velocity in the body frame:

$$\boldsymbol{\omega}_{\text{meas}} = \boldsymbol{\omega}_{\text{true}} + \mathbf{b}_g + \mathbf{n}_g$$

where $\mathbf{b}$ is a slowly varying **bias** and $\mathbf{n}$ is white noise.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
dt        = 0.005        # IMU timestep in seconds (200 Hz)
duration  = 20.0         # total trajectory duration (s)
speed     = 1.5          # figure-8 speed scale
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
n_steps = int(duration / dt)
t = np.arange(n_steps) * dt

# Generate a smooth figure-8 trajectory in 2D
freq = 2 * np.pi / duration
true_x = speed * np.sin(freq * t)
true_y = speed * np.sin(2 * freq * t) * 0.5

# True velocity (numerical derivative)
true_vx = np.gradient(true_x, dt)
true_vy = np.gradient(true_y, dt)

# True acceleration (numerical derivative of velocity)
true_ax = np.gradient(true_vx, dt)
true_ay = np.gradient(true_vy, dt)

# True heading angle and angular velocity
true_theta = np.arctan2(true_vy, true_vx)
# Unwrap to avoid discontinuities
true_theta = np.unwrap(true_theta)
true_omega = np.gradient(true_theta, dt)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(true_x, true_y, 'steelblue', lw=2)
axes[0].plot(true_x[0], true_y[0], 'o', color='forestgreen', ms=10, label='Start')
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_title('Ground Truth Trajectory (Figure 8)')
axes[0].set_aspect('equal'); axes[0].legend()

axes[1].plot(t, true_ax, 'steelblue', lw=1, label='$a_x$')
axes[1].plot(t, true_ay, 'tomato', lw=1, label='$a_y$')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Acceleration (m/s²)')
axes[1].set_title('True Acceleration'); axes[1].legend()

axes[2].plot(t, true_omega, 'forestgreen', lw=1)
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Angular velocity (rad/s)')
axes[2].set_title('True Angular Velocity')

plt.tight_layout(); plt.show()

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
accel_noise_std  = 0.05    # accelerometer white noise σ (m/s²)  try 0.01, 0.1
gyro_noise_std   = 0.005   # gyroscope white noise σ (rad/s)     try 0.001, 0.02
accel_bias       = np.array([0.02, -0.01])   # constant accel bias (m/s²)
gyro_bias        = 0.001   # constant gyro bias (rad/s)
# ─────────────────────────────────────────────────────────────────────────────

# Simulate noisy IMU readings
meas_ax = true_ax + accel_bias[0] + np.random.normal(0, accel_noise_std, n_steps)
meas_ay = true_ay + accel_bias[1] + np.random.normal(0, accel_noise_std, n_steps)
meas_omega = true_omega + gyro_bias + np.random.normal(0, gyro_noise_std, n_steps)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t, true_ax, 'k', lw=1.5, label='True $a_x$', alpha=0.8)
axes[0].plot(t, meas_ax, 'tomato', lw=0.3, alpha=0.6, label='Measured $a_x$')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('m/s²')
axes[0].set_title('Accelerometer X'); axes[0].legend(fontsize=8)

axes[1].plot(t, true_ay, 'k', lw=1.5, label='True $a_y$', alpha=0.8)
axes[1].plot(t, meas_ay, 'tomato', lw=0.3, alpha=0.6, label='Measured $a_y$')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('m/s²')
axes[1].set_title('Accelerometer Y'); axes[1].legend(fontsize=8)

axes[2].plot(t, true_omega, 'k', lw=1.5, label='True ω', alpha=0.8)
axes[2].plot(t, meas_omega, 'forestgreen', lw=0.3, alpha=0.6, label='Measured ω')
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('rad/s')
axes[2].set_title('Gyroscope'); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f'Accelerometer SNR (x): {np.std(true_ax) / accel_noise_std:.1f}')
print(f'Gyroscope SNR:         {np.std(true_omega) / gyro_noise_std:.1f}')

**What to notice:** The noise is hard to see when the signal is large, but during quiet periods (near zero acceleration) the noise dominates. The bias is invisible by eye; it shifts the whole signal up or down by a tiny amount. That tiny shift is the killer, as we will see next.

---
## 39.2 Bias and Drift: The Silent Killer

Two error sources create fundamentally different drift behaviors:

| Error Source | Position Error Growth | After 60 s |
|-------------|----------------------|------------|
| **White noise** (random walk) | $\propto t^{3/2}$ | moderate |
| **Constant bias** | $\propto t^2$ (quadratic!) | catastrophic |

A constant bias $b$ in the accelerometer creates a position error:
$$\Delta p = \frac{1}{2} b \, t^2$$

For $b = 0.01$ m/s$^2$: after 10 s $\Rightarrow$ 0.5 m, after 60 s $\Rightarrow$ 18 m, after 300 s $\Rightarrow$ 450 m.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
noise_std    = 0.02     # white noise standard deviation (m/s²)
bias_small   = 0.01     # small bias (m/s²)       try 0.005, 0.02
bias_large   = 0.1      # large bias (m/s²)       try 0.05, 0.2
sim_duration = 60.0     # simulation duration (s)  try 30, 120
sim_dt       = 0.01     # timestep (s)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(7)
N = int(sim_duration / sim_dt)
t_sim = np.arange(N) * sim_dt

def integrate_1d_accel(accel, dt, N):
    """Double integrate acceleration to get position."""
    vel = np.zeros(N)
    pos = np.zeros(N)
    for i in range(1, N):
        vel[i] = vel[i-1] + accel[i-1] * dt
        pos[i] = pos[i-1] + vel[i-1] * dt + 0.5 * accel[i-1] * dt**2
    return vel, pos

# Scenario (a): noise only (no bias)
accel_a = np.random.normal(0, noise_std, N)
vel_a, pos_a = integrate_1d_accel(accel_a, sim_dt, N)

# Scenario (b): small bias + noise
accel_b = bias_small + np.random.normal(0, noise_std, N)
vel_b, pos_b = integrate_1d_accel(accel_b, sim_dt, N)

# Scenario (c): large bias + noise
accel_c = bias_large + np.random.normal(0, noise_std, N)
vel_c, pos_c = integrate_1d_accel(accel_c, sim_dt, N)

# Analytical bias drift: p = 0.5 * b * t^2
pos_theory_b = 0.5 * bias_small * t_sim**2
pos_theory_c = 0.5 * bias_large * t_sim**2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t_sim, pos_a, 'steelblue', lw=1.5)
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Position error (m)')
axes[0].set_title(f'(a) Noise only (σ={noise_std})')

axes[1].plot(t_sim, pos_b, 'tomato', lw=1.5, label='Integrated')
axes[1].plot(t_sim, pos_theory_b, 'k--', lw=1, label=f'½bt² (b={bias_small})')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Position error (m)')
axes[1].set_title(f'(b) Small bias = {bias_small} m/s²')
axes[1].legend(fontsize=8)

axes[2].plot(t_sim, pos_c, 'orange', lw=1.5, label='Integrated')
axes[2].plot(t_sim, pos_theory_c, 'k--', lw=1, label=f'½bt² (b={bias_large})')
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Position error (m)')
axes[2].set_title(f'(c) Large bias = {bias_large} m/s²')
axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f'Position error at {sim_duration:.0f}s:')
print(f'  Noise only:  {pos_a[-1]:+.2f} m (random walk)')
print(f'  Small bias:  {pos_b[-1]:+.2f} m (theory: {pos_theory_b[-1]:.2f} m)')
print(f'  Large bias:  {pos_c[-1]:+.2f} m (theory: {pos_theory_c[-1]:.2f} m)')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_trials     = 50      # Monte Carlo runs for noise analysis
noise_level  = 0.02    # white noise σ (m/s²)
bias_level   = 0.01    # constant bias (m/s²)
mc_duration  = 120.0   # duration (s)
mc_dt        = 0.01    # timestep
# ─────────────────────────────────────────────────────────────────────────────

N_mc = int(mc_duration / mc_dt)
t_mc = np.arange(N_mc) * mc_dt

# Monte Carlo for noise-only drift
pos_noise_trials = np.zeros((n_trials, N_mc))
for trial in range(n_trials):
    a_noise = np.random.normal(0, noise_level, N_mc)
    _, pos_noise_trials[trial] = integrate_1d_accel(a_noise, mc_dt, N_mc)

rms_noise = np.sqrt(np.mean(pos_noise_trials**2, axis=0))

# Bias drift (deterministic)
pos_bias_drift = 0.5 * bias_level * t_mc**2

# Theoretical curves
# White noise position error RMS ~ σ * dt^0.5 * t^1.5 / sqrt(3)
# (from triple integration of white noise)
t_theory = t_mc[1:]  # avoid log(0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(t_mc[1:], rms_noise[1:], 'steelblue', lw=2, label=f'White noise RMS (σ={noise_level})')
ax.loglog(t_mc[1:], pos_bias_drift[1:], 'tomato', lw=2, label=f'Bias drift (b={bias_level})')

# Reference slopes
t_ref = np.logspace(0, np.log10(mc_duration), 50)
ax.loglog(t_ref, 0.003 * t_ref**1.5, 'steelblue', ls='--', alpha=0.5, label='$\\propto t^{3/2}$ (noise)')
ax.loglog(t_ref, 0.5 * bias_level * t_ref**2, 'tomato', ls='--', alpha=0.5, label='$\\propto t^{2}$ (bias)')

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Position error (m)', fontsize=12)
ax.set_title('Noise vs Bias: Position Error Growth (Log Log Scale)', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# Find crossover time
crossover_idx = np.argmax(pos_bias_drift[1:] > rms_noise[1:])
if crossover_idx > 0:
    print(f'Bias dominates noise after ~{t_mc[crossover_idx+1]:.1f} s')
print(f'At t=10s:  noise={rms_noise[int(10/mc_dt)]:.3f} m,  bias={pos_bias_drift[int(10/mc_dt)]:.3f} m')
print(f'At t=60s:  noise={rms_noise[int(60/mc_dt)]:.3f} m,  bias={pos_bias_drift[int(60/mc_dt)]:.3f} m')
print(f'At t=120s: noise={rms_noise[-1]:.3f} m,  bias={pos_bias_drift[-1]:.3f} m')

**Key insight:** On the log-log plot, noise grows with slope 3/2 and bias grows with slope 2. The bias *always* wins eventually. For consumer IMUs, bias dominates within seconds. For tactical grade IMUs, it may take minutes. For navigation grade IMUs (submarines, spacecraft), hours. But it always wins.

---
## 39.3 Integration: From IMU Readings to Trajectory

The full IMU integration pipeline:

1. **Gyroscope** $\rightarrow$ integrate angular velocity $\rightarrow$ **orientation** $R(t)$
2. **Accelerometer** in body frame $\rightarrow$ rotate to world frame: $\mathbf{a}^w = R \, \mathbf{a}^b$
3. Subtract gravity: $\mathbf{a}_{\text{net}} = \mathbf{a}^w - \mathbf{g}$
4. Integrate once $\rightarrow$ **velocity**
5. Integrate twice $\rightarrow$ **position**

In 2D, the rotation matrix is:
$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
imu_dt           = 0.005    # IMU timestep (s), 200 Hz     try 0.01, 0.001
pipeline_dur     = 20.0     # duration (s)
a_noise          = 0.05     # accelerometer noise σ (m/s²)
g_noise          = 0.005    # gyroscope noise σ (rad/s)
a_bias_val       = 0.01     # accelerometer bias magnitude
g_bias_val       = 0.0005   # gyroscope bias (rad/s)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
N_p = int(pipeline_dur / imu_dt)
t_p = np.arange(N_p) * imu_dt

# Ground truth: figure-8
freq_p = 2 * np.pi / pipeline_dur
gt_x = 3.0 * np.sin(freq_p * t_p)
gt_y = 1.5 * np.sin(2 * freq_p * t_p)
gt_vx = np.gradient(gt_x, imu_dt)
gt_vy = np.gradient(gt_y, imu_dt)
gt_ax = np.gradient(gt_vx, imu_dt)
gt_ay = np.gradient(gt_vy, imu_dt)
gt_theta = np.unwrap(np.arctan2(gt_vy, gt_vx))
gt_omega = np.gradient(gt_theta, imu_dt)

# In a 2D planar scenario, the accelerometer measures body-frame acceleration.
# Body frame accel = R^T * world_accel (no gravity in 2D horizontal plane)
gt_ax_body = np.zeros(N_p)
gt_ay_body = np.zeros(N_p)
for i in range(N_p):
    c, s = np.cos(gt_theta[i]), np.sin(gt_theta[i])
    gt_ax_body[i] =  c * gt_ax[i] + s * gt_ay[i]
    gt_ay_body[i] = -s * gt_ax[i] + c * gt_ay[i]

# Add noise and bias to body-frame measurements
meas_ax_b = gt_ax_body + a_bias_val + np.random.normal(0, a_noise, N_p)
meas_ay_b = gt_ay_body - a_bias_val * 0.5 + np.random.normal(0, a_noise, N_p)
meas_w = gt_omega + g_bias_val + np.random.normal(0, g_noise, N_p)

# === FULL IMU INTEGRATION PIPELINE ===
# Step 1: Integrate gyro to get orientation
est_theta = np.zeros(N_p)
est_theta[0] = gt_theta[0]  # known initial heading
for i in range(1, N_p):
    est_theta[i] = est_theta[i-1] + meas_w[i-1] * imu_dt

# Step 2 & 3: Rotate body accel to world frame (no gravity in 2D horizontal)
est_ax_w = np.zeros(N_p)
est_ay_w = np.zeros(N_p)
for i in range(N_p):
    c, s = np.cos(est_theta[i]), np.sin(est_theta[i])
    est_ax_w[i] = c * meas_ax_b[i] - s * meas_ay_b[i]
    est_ay_w[i] = s * meas_ax_b[i] + c * meas_ay_b[i]

# Step 4 & 5: Double integrate for velocity and position
est_vx = np.zeros(N_p); est_vy = np.zeros(N_p)
est_x  = np.zeros(N_p); est_y  = np.zeros(N_p)
est_vx[0] = gt_vx[0]; est_vy[0] = gt_vy[0]  # known initial velocity
est_x[0]  = gt_x[0];  est_y[0]  = gt_y[0]    # known initial position

for i in range(1, N_p):
    est_vx[i] = est_vx[i-1] + est_ax_w[i-1] * imu_dt
    est_vy[i] = est_vy[i-1] + est_ay_w[i-1] * imu_dt
    est_x[i]  = est_x[i-1]  + est_vx[i-1] * imu_dt
    est_y[i]  = est_y[i-1]  + est_vy[i-1] * imu_dt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(gt_x, gt_y, 'k', lw=2, label='Ground truth')
axes[0].plot(est_x, est_y, 'tomato', lw=1.5, label='IMU integrated')
axes[0].plot(gt_x[0], gt_y[0], 'o', color='forestgreen', ms=10)
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_title('Trajectory: IMU Integration'); axes[0].legend(fontsize=9)
axes[0].set_aspect('equal')

axes[1].plot(t_p, gt_theta, 'k', lw=1.5, label='True θ')
axes[1].plot(t_p, est_theta, 'steelblue', lw=1, label='Estimated θ')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Heading (rad)')
axes[1].set_title('Orientation (Gyro Integration)'); axes[1].legend(fontsize=9)

pos_err = np.sqrt((est_x - gt_x)**2 + (est_y - gt_y)**2)
axes[2].plot(t_p, pos_err, 'tomato', lw=1.5)
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Position error (m)')
axes[2].set_title('Position Error Over Time')

plt.tight_layout(); plt.show()
print(f'Final position error: {pos_err[-1]:.2f} m')
print(f'Final heading error:  {np.degrees(est_theta[-1] - gt_theta[-1]):.2f} deg')

**What to observe:** The trajectory starts well but gradually drifts away from ground truth. The heading error from gyro bias causes the accelerometer readings to be rotated into the wrong direction, creating lateral drift that compounds the position error.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
test_duration = 10.0       # test duration (s)
omega_const   = 1.0        # constant angular velocity (rad/s) for testing
dt_coarse     = 0.05       # coarse timestep for Euler      try 0.1, 0.01
dt_medium     = 0.01       # medium timestep for midpoint   try 0.05, 0.005
dt_fine       = 0.005      # fine timestep for RK4          try 0.01, 0.001
# ─────────────────────────────────────────────────────────────────────────────

def integrate_euler(omega_func, theta0, dt, T):
    """Euler method: θ_{k+1} = θ_k + ω_k * dt"""
    N = int(T / dt)
    theta = np.zeros(N)
    theta[0] = theta0
    t = np.arange(N) * dt
    for i in range(1, N):
        theta[i] = theta[i-1] + omega_func(t[i-1]) * dt
    return t, theta

def integrate_midpoint(omega_func, theta0, dt, T):
    """Midpoint method: θ_{k+1} = θ_k + ω(t_k + dt/2) * dt"""
    N = int(T / dt)
    theta = np.zeros(N)
    theta[0] = theta0
    t = np.arange(N) * dt
    for i in range(1, N):
        theta[i] = theta[i-1] + omega_func(t[i-1] + dt/2) * dt
    return t, theta

def integrate_rk4(omega_func, theta0, dt, T):
    """RK4 method for dθ/dt = ω(t)"""
    N = int(T / dt)
    theta = np.zeros(N)
    theta[0] = theta0
    t = np.arange(N) * dt
    for i in range(1, N):
        k1 = omega_func(t[i-1])
        k2 = omega_func(t[i-1] + dt/2)
        k3 = omega_func(t[i-1] + dt/2)
        k4 = omega_func(t[i-1] + dt)
        theta[i] = theta[i-1] + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)
    return t, theta

# Test case: sinusoidal angular velocity
# ω(t) = A * sin(2πf*t), so θ(t) = θ0 - A/(2πf) * cos(2πf*t) + A/(2πf)
A_omega = 2.0
f_omega = 0.5
omega_func = lambda t: A_omega * np.sin(2 * np.pi * f_omega * t)
theta_exact = lambda t: -A_omega / (2 * np.pi * f_omega) * np.cos(2 * np.pi * f_omega * t) + A_omega / (2 * np.pi * f_omega)

# Run all three methods at the SAME timestep for fair comparison
dt_compare = 0.05  # use a coarse dt to show differences

t_e, th_e = integrate_euler(omega_func, 0.0, dt_compare, test_duration)
t_m, th_m = integrate_midpoint(omega_func, 0.0, dt_compare, test_duration)
t_r, th_r = integrate_rk4(omega_func, 0.0, dt_compare, test_duration)

th_true_e = theta_exact(t_e)
th_true_m = theta_exact(t_m)
th_true_r = theta_exact(t_r)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(t_e, th_true_e, 'k', lw=2, label='Exact')
axes[0].plot(t_e, th_e, 'tomato', lw=1.5, ls='--', label=f'Euler (dt={dt_compare})')
axes[0].plot(t_m, th_m, 'steelblue', lw=1.5, ls='-.', label=f'Midpoint (dt={dt_compare})')
axes[0].plot(t_r, th_r, 'forestgreen', lw=1.5, ls=':', label=f'RK4 (dt={dt_compare})')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('θ (rad)')
axes[0].set_title('Integration Methods Comparison'); axes[0].legend(fontsize=8)

axes[1].semilogy(t_e, np.abs(th_e - th_true_e) + 1e-16, 'tomato', lw=1.5, label='Euler')
axes[1].semilogy(t_m, np.abs(th_m - th_true_m) + 1e-16, 'steelblue', lw=1.5, label='Midpoint')
axes[1].semilogy(t_r, np.abs(th_r - th_true_r) + 1e-16, 'forestgreen', lw=1.5, label='RK4')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('|Error| (rad)')
axes[1].set_title('Absolute Integration Error (log scale)'); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'Final error (Euler):    {np.abs(th_e[-1] - th_true_e[-1]):.6f} rad')
print(f'Final error (Midpoint): {np.abs(th_m[-1] - th_true_m[-1]):.6f} rad')
print(f'Final error (RK4):      {np.abs(th_r[-1] - th_true_r[-1]):.6f} rad')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
dt_values = [0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]  # timesteps to test
conv_duration = 10.0
# ─────────────────────────────────────────────────────────────────────────────

errors_euler = []
errors_mid   = []
errors_rk4   = []

for dt_test in dt_values:
    t_e, th_e = integrate_euler(omega_func, 0.0, dt_test, conv_duration)
    t_m, th_m = integrate_midpoint(omega_func, 0.0, dt_test, conv_duration)
    t_r, th_r = integrate_rk4(omega_func, 0.0, dt_test, conv_duration)
    errors_euler.append(np.abs(th_e[-1] - theta_exact(t_e[-1])))
    errors_mid.append(np.abs(th_m[-1] - theta_exact(t_m[-1])))
    errors_rk4.append(np.abs(th_r[-1] - theta_exact(t_r[-1])))

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(dt_values, errors_euler, 'o-', color='tomato', lw=2, ms=7, label='Euler (order 1)')
ax.loglog(dt_values, errors_mid, 's-', color='steelblue', lw=2, ms=7, label='Midpoint (order 2)')
ax.loglog(dt_values, errors_rk4, '^-', color='forestgreen', lw=2, ms=7, label='RK4 (order 4)')

# Reference slopes
dt_ref = np.array(dt_values)
ax.loglog(dt_ref, 0.5 * dt_ref, 'tomato', ls=':', alpha=0.4, label='$\\propto \\Delta t$')
ax.loglog(dt_ref, 2 * dt_ref**2, 'steelblue', ls=':', alpha=0.4, label='$\\propto \\Delta t^2$')
ax.loglog(dt_ref, 10 * dt_ref**4, 'forestgreen', ls=':', alpha=0.4, label='$\\propto \\Delta t^4$')

ax.set_xlabel('Timestep Δt (s)', fontsize=12)
ax.set_ylabel('Final angle error (rad)', fontsize=12)
ax.set_title('Integration Order Convergence', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Key takeaway:** Euler integration is first order (halving $\Delta t$ halves the error). Midpoint is second order (halving $\Delta t$ quarters the error). RK4 is fourth order (halving $\Delta t$ reduces error by 16x). In practice, IMUs run at 200+ Hz, so even Euler is usually adequate. The real enemy is bias, not integration order.

---
## 39.4 Bias Estimation and Calibration

If the IMU is **stationary** for a few seconds, we can estimate the bias. During stationary periods:
- True acceleration = 0 (in horizontal plane)
- True angular velocity = 0

So the mean of the measurements equals the bias:
$$\hat{b}_a = \frac{1}{N_{\text{cal}}} \sum_{k=1}^{N_{\text{cal}}} a_{\text{meas},k}, \qquad \hat{b}_g = \frac{1}{N_{\text{cal}}} \sum_{k=1}^{N_{\text{cal}}} \omega_{\text{meas},k}$$

The estimation error decreases as $\sigma / \sqrt{N_{\text{cal}}}$, so longer calibration gives better bias estimates.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
cal_duration   = 5.0      # stationary calibration period (s)  try 1.0, 3.0, 10.0
cal_dt         = 0.005    # timestep during calibration
true_a_bias    = np.array([0.015, -0.008])   # true accelerometer bias
true_g_bias    = 0.0008   # true gyroscope bias (rad/s)
cal_a_noise    = 0.05     # accelerometer noise σ
cal_g_noise    = 0.005    # gyroscope noise σ
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(99)
N_cal = int(cal_duration / cal_dt)

# Stationary IMU readings: true signal is zero, only bias + noise
cal_ax = true_a_bias[0] + np.random.normal(0, cal_a_noise, N_cal)
cal_ay = true_a_bias[1] + np.random.normal(0, cal_a_noise, N_cal)
cal_w  = true_g_bias + np.random.normal(0, cal_g_noise, N_cal)

# Estimate bias as running mean
est_bias_ax = np.cumsum(cal_ax) / np.arange(1, N_cal + 1)
est_bias_ay = np.cumsum(cal_ay) / np.arange(1, N_cal + 1)
est_bias_w  = np.cumsum(cal_w)  / np.arange(1, N_cal + 1)

t_cal = np.arange(N_cal) * cal_dt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t_cal, est_bias_ax, 'steelblue', lw=1.5)
axes[0].axhline(true_a_bias[0], color='k', ls='--', lw=1, label=f'True bias = {true_a_bias[0]}')
axes[0].fill_between(t_cal, 
    true_a_bias[0] - cal_a_noise / np.sqrt(np.arange(1, N_cal+1)),
    true_a_bias[0] + cal_a_noise / np.sqrt(np.arange(1, N_cal+1)),
    alpha=0.2, color='steelblue', label='±σ/√N')
axes[0].set_xlabel('Calibration time (s)'); axes[0].set_ylabel('Estimated bias (m/s²)')
axes[0].set_title('Accel X Bias Estimation'); axes[0].legend(fontsize=8)

axes[1].plot(t_cal, est_bias_ay, 'tomato', lw=1.5)
axes[1].axhline(true_a_bias[1], color='k', ls='--', lw=1, label=f'True bias = {true_a_bias[1]}')
axes[1].fill_between(t_cal,
    true_a_bias[1] - cal_a_noise / np.sqrt(np.arange(1, N_cal+1)),
    true_a_bias[1] + cal_a_noise / np.sqrt(np.arange(1, N_cal+1)),
    alpha=0.2, color='tomato', label='±σ/√N')
axes[1].set_xlabel('Calibration time (s)'); axes[1].set_ylabel('Estimated bias (m/s²)')
axes[1].set_title('Accel Y Bias Estimation'); axes[1].legend(fontsize=8)

axes[2].plot(t_cal, np.degrees(est_bias_w), 'forestgreen', lw=1.5)
axes[2].axhline(np.degrees(true_g_bias), color='k', ls='--', lw=1, label=f'True bias = {np.degrees(true_g_bias):.3f}°/s')
axes[2].fill_between(t_cal,
    np.degrees(true_g_bias - cal_g_noise / np.sqrt(np.arange(1, N_cal+1))),
    np.degrees(true_g_bias + cal_g_noise / np.sqrt(np.arange(1, N_cal+1))),
    alpha=0.2, color='forestgreen', label='±σ/√N')
axes[2].set_xlabel('Calibration time (s)'); axes[2].set_ylabel('Estimated bias (deg/s)')
axes[2].set_title('Gyro Bias Estimation'); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

final_est_a = np.array([est_bias_ax[-1], est_bias_ay[-1]])
final_est_g = est_bias_w[-1]
print(f'True accel bias:      [{true_a_bias[0]:.4f}, {true_a_bias[1]:.4f}] m/s²')
print(f'Estimated accel bias: [{final_est_a[0]:.4f}, {final_est_a[1]:.4f}] m/s²')
print(f'True gyro bias:       {true_g_bias:.5f} rad/s ({np.degrees(true_g_bias):.4f} deg/s)')
print(f'Estimated gyro bias:  {final_est_g:.5f} rad/s ({np.degrees(final_est_g):.4f} deg/s)')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# Show improvement from bias calibration on the figure-8 trajectory
calib_time  = 3.0     # calibration period (s)   try 1.0, 5.0, 10.0
motion_time = 20.0    # motion duration (s)
imp_dt      = 0.005   # timestep
imp_a_noise = 0.05
imp_g_noise = 0.005
imp_a_bias  = np.array([0.015, -0.008])
imp_g_bias  = 0.0008
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(77)

# Phase 1: Stationary calibration
N_calib = int(calib_time / imp_dt)
calib_ax = imp_a_bias[0] + np.random.normal(0, imp_a_noise, N_calib)
calib_ay = imp_a_bias[1] + np.random.normal(0, imp_a_noise, N_calib)
calib_w  = imp_g_bias + np.random.normal(0, imp_g_noise, N_calib)

est_ab = np.array([np.mean(calib_ax), np.mean(calib_ay)])
est_gb = np.mean(calib_w)

# Phase 2: Motion (figure-8)
N_mot = int(motion_time / imp_dt)
t_mot = np.arange(N_mot) * imp_dt
freq_m = 2 * np.pi / motion_time
gt_xm = 3.0 * np.sin(freq_m * t_mot)
gt_ym = 1.5 * np.sin(2 * freq_m * t_mot)
gt_vxm = np.gradient(gt_xm, imp_dt)
gt_vym = np.gradient(gt_ym, imp_dt)
gt_axm = np.gradient(gt_vxm, imp_dt)
gt_aym = np.gradient(gt_vym, imp_dt)
gt_thm = np.unwrap(np.arctan2(gt_vym, gt_vxm))
gt_wm  = np.gradient(gt_thm, imp_dt)

# Body frame true accelerations
gt_axb = np.zeros(N_mot); gt_ayb = np.zeros(N_mot)
for i in range(N_mot):
    c, s = np.cos(gt_thm[i]), np.sin(gt_thm[i])
    gt_axb[i] =  c * gt_axm[i] + s * gt_aym[i]
    gt_ayb[i] = -s * gt_axm[i] + c * gt_aym[i]

# Noisy IMU measurements
m_axb = gt_axb + imp_a_bias[0] + np.random.normal(0, imp_a_noise, N_mot)
m_ayb = gt_ayb + imp_a_bias[1] + np.random.normal(0, imp_a_noise, N_mot)
m_w   = gt_wm  + imp_g_bias + np.random.normal(0, imp_g_noise, N_mot)

def integrate_imu_2d(ax_b, ay_b, omega, dt, x0, y0, vx0, vy0, theta0):
    """Full 2D IMU integration pipeline."""
    N = len(ax_b)
    theta = np.zeros(N); theta[0] = theta0
    vx = np.zeros(N); vy = np.zeros(N); vx[0] = vx0; vy[0] = vy0
    x = np.zeros(N); y = np.zeros(N); x[0] = x0; y[0] = y0
    for i in range(1, N):
        theta[i] = theta[i-1] + omega[i-1] * dt
        c, s = np.cos(theta[i-1]), np.sin(theta[i-1])
        aw_x = c * ax_b[i-1] - s * ay_b[i-1]
        aw_y = s * ax_b[i-1] + c * ay_b[i-1]
        vx[i] = vx[i-1] + aw_x * dt
        vy[i] = vy[i-1] + aw_y * dt
        x[i] = x[i-1] + vx[i-1] * dt
        y[i] = y[i-1] + vy[i-1] * dt
    return x, y, theta

# Without bias compensation
x_raw, y_raw, _ = integrate_imu_2d(
    m_axb, m_ayb, m_w, imp_dt,
    gt_xm[0], gt_ym[0], gt_vxm[0], gt_vym[0], gt_thm[0])

# With bias compensation
x_cal, y_cal, _ = integrate_imu_2d(
    m_axb - est_ab[0], m_ayb - est_ab[1], m_w - est_gb, imp_dt,
    gt_xm[0], gt_ym[0], gt_vxm[0], gt_vym[0], gt_thm[0])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(gt_xm, gt_ym, 'k', lw=2.5, label='Ground truth')
axes[0].plot(x_raw, y_raw, 'tomato', lw=1.5, alpha=0.8, label='No calibration')
axes[0].plot(x_cal, y_cal, 'steelblue', lw=1.5, alpha=0.8, label=f'With {calib_time}s calibration')
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_title('Trajectory: Effect of Bias Calibration')
axes[0].legend(fontsize=9); axes[0].set_aspect('equal')

err_raw = np.sqrt((x_raw - gt_xm)**2 + (y_raw - gt_ym)**2)
err_cal = np.sqrt((x_cal - gt_xm)**2 + (y_cal - gt_ym)**2)
axes[1].plot(t_mot, err_raw, 'tomato', lw=1.5, label='No calibration')
axes[1].plot(t_mot, err_cal, 'steelblue', lw=1.5, label=f'With {calib_time}s calibration')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Position error (m)')
axes[1].set_title('Position Error Comparison'); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'Final position error (no calibration):   {err_raw[-1]:.2f} m')
print(f'Final position error (with calibration): {err_cal[-1]:.2f} m')
print(f'Improvement factor: {err_raw[-1] / max(err_cal[-1], 1e-6):.1f}x')

**Important caveat:** Bias calibration helps enormously but does not eliminate drift entirely. The residual drift comes from: (1) noise that cannot be calibrated away, (2) bias instability (the bias slowly changes over time), and (3) errors in the bias estimate itself. For long duration navigation, you still need external corrections (GPS, visual landmarks, etc.).

---
## Capstone: Dead Reckoning a Square Path

A robot drives a square: four straight segments (2 m each) connected by four 90 degree left turns. We simulate IMU readings with noise and bias, integrate to recover the trajectory, then apply stationary bias calibration and compare.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
side_length    = 2.0      # square side length (m)
drive_speed    = 0.5      # forward speed (m/s)
turn_rate      = np.pi/2  # rad/s during turns (takes 1 s for 90 deg)
cap_dt         = 0.005    # IMU timestep
cap_a_noise    = 0.04     # accelerometer noise σ
cap_g_noise    = 0.003    # gyroscope noise σ
cap_a_bias_x   = 0.012    # accel bias x
cap_a_bias_y   = -0.006   # accel bias y
cap_g_bias     = 0.0006   # gyro bias (rad/s)
cal_period     = 3.0      # stationary calibration period (s)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(123)

# Build the ground truth square trajectory
drive_time = side_length / drive_speed  # time per straight segment
turn_time  = (np.pi / 2) / turn_rate    # time per turn

# Sequence: straight, turn, straight, turn, ...
segments = []
for _ in range(4):
    N_drive = int(drive_time / cap_dt)
    N_turn  = int(turn_time / cap_dt)
    # Straight: constant velocity, no angular rate
    segments.append(('drive', N_drive))
    segments.append(('turn', N_turn))

# Build true acceleration (body frame) and angular velocity
total_N = sum(n for _, n in segments)
# Add calibration period at the start
N_cal_cap = int(cal_period / cap_dt)
total_with_cal = N_cal_cap + total_N

true_ab_x = np.zeros(total_with_cal)  # body x accel (forward)
true_ab_y = np.zeros(total_with_cal)  # body y accel (lateral)
true_w_cap = np.zeros(total_with_cal)  # angular velocity

# During calibration: everything is zero (stationary)
# During motion: build from segments
idx = N_cal_cap
for seg_type, seg_N in segments:
    if seg_type == 'turn':
        true_w_cap[idx:idx+seg_N] = turn_rate
    # For 'drive': acceleration is zero (constant velocity)
    # Note: in reality there is a brief acceleration at start/stop
    # but for simplicity we assume instantaneous velocity changes
    idx += seg_N

# Build ground truth trajectory by integration
gt_th_cap = np.zeros(total_with_cal)
gt_vx_cap = np.zeros(total_with_cal)
gt_vy_cap = np.zeros(total_with_cal)
gt_x_cap  = np.zeros(total_with_cal)
gt_y_cap  = np.zeros(total_with_cal)

# After calibration, start moving forward
idx = N_cal_cap
for seg_type, seg_N in segments:
    for j in range(seg_N):
        if idx + j >= total_with_cal:
            break
        k = idx + j
        if k > 0:
            gt_th_cap[k] = gt_th_cap[k-1] + true_w_cap[k-1] * cap_dt
        if seg_type == 'drive':
            gt_vx_cap[k] = drive_speed * np.cos(gt_th_cap[k])
            gt_vy_cap[k] = drive_speed * np.sin(gt_th_cap[k])
        else:
            gt_vx_cap[k] = 0.0
            gt_vy_cap[k] = 0.0
        if k > 0:
            gt_x_cap[k] = gt_x_cap[k-1] + gt_vx_cap[k-1] * cap_dt
            gt_y_cap[k] = gt_y_cap[k-1] + gt_vy_cap[k-1] * cap_dt
    idx += seg_N

# Generate noisy IMU measurements
m_axb_cap = true_ab_x + cap_a_bias_x + np.random.normal(0, cap_a_noise, total_with_cal)
m_ayb_cap = true_ab_y + cap_a_bias_y + np.random.normal(0, cap_a_noise, total_with_cal)
m_w_cap   = true_w_cap + cap_g_bias + np.random.normal(0, cap_g_noise, total_with_cal)

# Also add body-frame acceleration from velocity changes (centripetal during turns)
# During turns at constant speed, centripetal accel = v²/r = v * ω
idx = N_cal_cap
for seg_type, seg_N in segments:
    if seg_type == 'drive':
        # Moving forward at constant speed: true accel is zero
        pass
    idx += seg_N

print(f'Total simulation: {total_with_cal * cap_dt:.1f} s ({N_cal_cap * cap_dt:.1f} s cal + {total_N * cap_dt:.1f} s motion)')
print(f'Segments: 4 drives of {drive_time:.1f} s, 4 turns of {turn_time:.1f} s')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# (uses data from previous cell)
# ─────────────────────────────────────────────────────────────────────────────

# Integrate WITHOUT bias compensation (skip calibration data, use raw IMU)
motion_start = N_cal_cap
m_axb_mot = m_axb_cap[motion_start:]
m_ayb_mot = m_ayb_cap[motion_start:]
m_w_mot   = m_w_cap[motion_start:]
N_mot_cap = len(m_axb_mot)

x_nc, y_nc, th_nc = integrate_imu_2d(
    m_axb_mot, m_ayb_mot, m_w_mot, cap_dt,
    0, 0, drive_speed, 0, 0)  # start heading east at drive_speed

# Estimate bias from calibration period
est_abx = np.mean(m_axb_cap[:motion_start])
est_aby = np.mean(m_ayb_cap[:motion_start])
est_gb_cap = np.mean(m_w_cap[:motion_start])

# Integrate WITH bias compensation
x_wc, y_wc, th_wc = integrate_imu_2d(
    m_axb_mot - est_abx, m_ayb_mot - est_aby, m_w_mot - est_gb_cap, cap_dt,
    0, 0, drive_speed, 0, 0)

# Ground truth for motion phase
gt_x_mot = gt_x_cap[motion_start:]
gt_y_mot = gt_y_cap[motion_start:]
t_mot_cap = np.arange(N_mot_cap) * cap_dt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(gt_x_mot, gt_y_mot, 'k', lw=3, label='Ground truth', zorder=5)
axes[0].plot(x_nc, y_nc, 'tomato', lw=1.5, alpha=0.8, label='No calibration')
axes[0].plot(x_wc, y_wc, 'steelblue', lw=1.5, alpha=0.8, label='With calibration')
axes[0].plot(0, 0, 'o', color='forestgreen', ms=10, zorder=6)
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_title('Dead Reckoning: Square Path')
axes[0].legend(fontsize=9); axes[0].set_aspect('equal')

err_nc = np.sqrt((x_nc - gt_x_mot)**2 + (y_nc - gt_y_mot)**2)
err_wc = np.sqrt((x_wc - gt_x_mot)**2 + (y_wc - gt_y_mot)**2)

axes[1].plot(t_mot_cap, err_nc, 'tomato', lw=1.5, label='No calibration')
axes[1].plot(t_mot_cap, err_wc, 'steelblue', lw=1.5, label='With calibration')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Position error (m)')
axes[1].set_title('Position Error Over Time'); axes[1].legend(fontsize=9)

# Heading error
gt_th_mot = gt_th_cap[motion_start:]
axes[2].plot(t_mot_cap, np.degrees(th_nc - gt_th_mot), 'tomato', lw=1.5, label='No cal')
axes[2].plot(t_mot_cap, np.degrees(th_wc - gt_th_mot), 'steelblue', lw=1.5, label='With cal')
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Heading error (deg)')
axes[2].set_title('Heading Error Over Time'); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'\n=== Capstone Results ===')
print(f'Bias estimation ({cal_period}s calibration):')
print(f'  Accel X: true={cap_a_bias_x:.4f}, est={est_abx:.4f}, error={abs(est_abx-cap_a_bias_x):.5f} m/s²')
print(f'  Accel Y: true={cap_a_bias_y:.4f}, est={est_aby:.4f}, error={abs(est_aby-cap_a_bias_y):.5f} m/s²')
print(f'  Gyro:    true={cap_g_bias:.5f}, est={est_gb_cap:.5f}, error={abs(est_gb_cap-cap_g_bias):.6f} rad/s')
print(f'\nFinal position error:')
print(f'  Without calibration: {err_nc[-1]:.3f} m')
print(f'  With calibration:    {err_wc[-1]:.3f} m')
print(f'  Improvement:         {err_nc[-1]/max(err_wc[-1],1e-6):.1f}x')

**Capstone takeaways:**
1. Without calibration, the square path drifts significantly; the robot does not return to the start.
2. A brief stationary period (3 seconds) allows bias estimation that dramatically reduces drift.
3. Even with calibration, residual drift accumulates because of noise and bias instability.
4. For long duration operation, external measurements (GPS, cameras, magnetometer) are essential to bound the drift. This is exactly the motivation for **visual inertial fusion** (Chapter 40).

---
## Exercises

### Exercise 39.1: Gravity in 3D

In 2D horizontal motion, gravity does not appear in the accelerometer reading (it points out of the plane). In 3D, the accelerometer reads $\mathbf{a}_{\text{meas}} = R^\top(\mathbf{a}_{\text{true}} - \mathbf{g})$. Simulate a 3D IMU at rest. The accelerometer should read $[0, 0, +9.81]$ m/s$^2$ (pointing up in the body frame). Add tilt: rotate the body by 5 degrees around the x axis. What does the accelerometer read now? Verify that the magnitude is still $g$.

In [ ]:
# Your code here

### Exercise 39.2: Bias Random Walk

Real IMU biases are not constant; they drift slowly over time (modeled as a random walk). Modify the simulation to use a time varying bias: $b_{k+1} = b_k + w_b$ where $w_b \sim \mathcal{N}(0, \sigma_b^2 \Delta t)$. Set $\sigma_b = 0.001$ m/s$^2$/s$^{1/2}$. How does the position drift compare to a constant bias of the same initial value? Plot both on the same axes.

In [ ]:
# Your code here

### Exercise 39.3: Allan Variance

**Allan variance** is the standard method for characterizing IMU noise. Given a long stationary recording of $N$ samples at rate $f_s$, compute the Allan variance at cluster time $\tau = m / f_s$:

$$\sigma^2_{\text{Allan}}(\tau) = \frac{1}{2(N - 2m)} \sum_{k=1}^{N-2m} \left( \bar{y}_{k+m} - \bar{y}_k \right)^2$$

where $\bar{y}_k = \frac{1}{m} \sum_{j=k}^{k+m-1} y_j$ is the cluster average.

Generate 100,000 samples of stationary gyroscope data with white noise ($\sigma = 0.01$ rad/s) and a random walk bias ($\sigma_b = 10^{-4}$ rad/s/$\sqrt{\text{s}}$). Compute and plot the Allan deviation ($\sqrt{\sigma^2_{\text{Allan}}}$) vs $\tau$ on a log log plot. You should see a slope of $-1/2$ for short $\tau$ (white noise region) and $+1/2$ for long $\tau$ (random walk region).

In [ ]:
# Your code here

### Exercise 39.4: Temperature Dependence

IMU biases change with temperature. Model a temperature sweep from 20C to 40C over 600 seconds with bias $b(T) = b_0 + k (T - T_{\text{ref}})$ where $k = 0.002$ m/s$^2$/C. Simulate and integrate. Then implement a **temperature compensation** lookup table: record bias at several temperatures, interpolate during operation. Show the improvement.

In [ ]:
# Your code here

### Exercise 39.5: IMU Grade Comparison

Different IMU grades have very different noise and bias characteristics:

| Grade | Gyro bias stability | Accel bias stability | Example |
|-------|-------------------|---------------------|----------|
| Consumer | 10 deg/hr | 1 mg | Phone IMU |
| Tactical | 1 deg/hr | 0.1 mg | Drone IMU |
| Navigation | 0.01 deg/hr | 0.01 mg | Submarine INS |

(1 mg = 0.00981 m/s$^2$, 1 deg/hr = 4.85 $\times$ 10$^{-6}$ rad/s)

Simulate all three grades performing dead reckoning on the square path for 60 seconds. Plot the position error for each. At what time does each grade exceed 1 m of error?

In [ ]:
# Your code here